In [45]:
# init
import requests
import pickle
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsPandas.helpers as tph
import os
import boto3
import pandas as pd
import copy

import shutil

def pckgs_reload():
    reload(tgm)
    reload(tph)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
SCRAPE_DIR = DATA_DIR / 'raw/osm countries queries'

## Review state of tasks

In [46]:
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

task_state = {
    'scrape':{'dir':'data/raw/osm countries queries', 'required_task':'scrape'},
    'clean':{'dir':'data/cleaned', 'required_task':'scrape'},
    'test_basic':{'dir':'data/tests results/osm basic test','required_task':'clean'}
}

tasks = ['scrape', 'clean', 'test_basic']
for task in tasks:
    task_path = ROOT / task_state[task]['dir']
    ok = [c for c, val in process_state.items() if (val[task]['status'] in ['ok', 'error'])]
    pending = [c for c, val in process_state.items() if (val[task]['status'] in ['pending', 'error'])]
    countries_dirs = [dir.relative_to(task_path).name for dir in task_path.glob('*') if dir.is_dir()]
    
    task_state[task]['ok'] = ok
    task_state[task]['pending'] = pending
    previous_task_ok = task_state[task_state[task]['required_task']]['ok']
    task_state[task]['to_process'] = tgm.complement(previous_task_ok, ok)
    task_state[task]['local_dirs'] = countries_dirs
    task_state[task]['ok_wihout_local_dirs'] = tgm.complement(ok, countries_dirs)
    previous_task_local_dirs = task_state[task_state[task]['required_task']]['local_dirs']
    task_state[task]['to_process_wihout_required_local_dirs'] = tgm.complement(tgm.complement(previous_task_ok, ok), previous_task_local_dirs)

# convert to length
task_state_resume = copy.deepcopy(task_state)
for task,val in task_state.items():
    for k,v in task_state_resume[task].items():
        if k in ['dir', 'required_task']: continue
        task_state_resume[task][k] = len(v)

214


In [47]:
pd.DataFrame(task_state_resume).T

,dir,required_task,ok,pending,to_process,local_dirs,ok_wihout_local_dirs,to_process_wihout_required_local_dirs
scrape,data/raw/osm countries queries,scrape,83,131,0,83,0,0
clean,data/cleaned,scrape,83,131,0,83,0,0
test_basic,data/tests results/osm basic test,clean,0,214,83,72,0,0


# Fix process state

## * initial state

In [90]:
osmMetaCountrDict = tgm.load(DATA_DIR / 'osmMetaCountrDict.json')
print(len(osmMetaCountrDict))

241


In [91]:
process_state = {}
for country in osmMetaCountrDict.keys():
    process_state[country] = {
        "scrape": {
            "status": "pending",
            "type": 'normal',
            "last_run": None,
            "error": None
        },
        "clean": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_basic": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_first_level": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_duplicates": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "fix": {
            "status": "pending",
            "last_run": None,
            "error": None
        }
    }
print(len(process_state))

241


In [154]:
tgm.dump(DATA_DIR / 'process_state.json', process_state)

## * update state

### * scrape

In [166]:
countries_dirs = [dir for dir in (DATA_DIR / 'raw/osm countries queries').glob('*') if dir.is_dir()]
print(len(countries_dirs))

97


In [33]:
chunks_countries = []
for country_dir in countries_dirs:
    if country_dir.is_dir():
        files = [f for f in country_dir.glob('*')]
        if len(files) > 1:
            chunks_countries.append(country_dir.name)
chunks_countries            

['Argentina', 'Peru', 'UnitedStates']

In [95]:
import datetime
for country in processed_countries:
    process_state[country]['scrape']['status'] = 'ok'
    process_state[country]['scrape']['last_run'] = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
    if country in chunks_countries:
        state = tgm.load(DATA_DIR / 'raw/osm countries queries' / country / 'state.pkl')
        process_state[country]['scrape']['type'] = 'chunk'
        process_state[country]['scrape']['chunk_state'] = state

In [97]:
process_state['Argentina']

{'scrape': {'status': 'ok',
  'type': 'chunk',
  'last_run': '2025-12-02T16:06:44',
  'error': None,
  'chunk_state': {'2': {'processed': {'286393'},
    'failed': set(),
    'discovered': {'286393'},
    'next_chunk_index': 1},
   '4': {'processed': {'153536',
     '153538',
     '153539',
     '153540',
     '153541',
     '153543',
     '153544',
     '153545',
     '153547',
     '153548',
     '153549',
     '153550',
     '153551',
     '153552',
     '153553',
     '153554',
     '153556',
     '153558',
     '1606727',
     '1632167',
     '2405230',
     '2849847',
     '301542',
     '3082668',
     '3592494'},
    'failed': set(),
    'discovered': {'153536',
     '153538',
     '153539',
     '153540',
     '153541',
     '153543',
     '153544',
     '153545',
     '153547',
     '153548',
     '153549',
     '153550',
     '153551',
     '153552',
     '153553',
     '153554',
     '153556',
     '153558',
     '1606727',
     '1632167',
     '2405230',
     '2849847',
  

In [98]:
tgm.dump(DATA_DIR / 'process_state.json', process_state)

### * clean

In [177]:
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

241


In [191]:
clean_countries = [dir.name for dir in (DATA_DIR / 'normalized').glob('*') if dir.is_dir()]
print(len(clean_countries))

72


In [194]:
scrape_countries = [c for c,v in process_state.items() if v['scrape']['status'] == 'ok']
print(len(scrape_countries))

98


In [195]:
print(len(tgm.complement(scrape_countries, clean_countries)))
print(len(tgm.complement(clean_countries, scrape_countries)))

26
0


In [197]:
import datetime
for country in clean_countries:
    process_state[country]['clean']['status'] = 'ok'
    process_state[country]['clean']['last_run'] = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

In [198]:
tgm.dump(DATA_DIR / 'process_state.json', process_state)

## * general review

In [156]:
# update from automated repo
shutil.copy2(AUTOMATED_REPO_DIR / 'data' / 'process_state.json', DATA_DIR / 'process_state.json')

WindowsPath('c:/Users/gonta/D/study/full stack/projects/administrative divisions osm scrape/data/process_state.json')

In [200]:
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

241


In [209]:
temp = [c for c, val in process_state.items() if (val['scrape']['status'] == 'ok') and val['clean']['status'] == 'pending']
print(len(temp))

26


In [201]:
import copy
process_state_resume = copy.deepcopy(process_state)
for country, val in process_state_resume.items():
    if val['scrape']['type'] == 'chunk':
        val['scrape']['chunk_state'] = {k:{k2:(len(v2) if type(v2) == set else v2) for k2,v2 in val.items()} for k,val in val['scrape']['chunk_state'].items()}
    process_state_resume[country] = val

In [202]:
rows = []
for country, sections in process_state_resume.items():
    for section_name, info in sections.items():
        rows.append({
            "country": country,
            "task": section_name,
            **info
        })

df = pd.DataFrame(rows)
df = df.sort_values(by=['country','task'])

In [105]:
tph.to_df(process_state_resume['Armenia']).df_peek()

,status,type,last_run,error
scrape,pending,normal,None,None
clean,pending,NaN,None,None
test_basic,pending,NaN,None,None
test_first_level,pending,NaN,None,None
test_duplicates,pending,NaN,None,None
fix,pending,NaN,None,None


In [106]:
tph.to_df(process_state_resume['Argentina']).df_peek()

,status,type,last_run,error,chunk_state
scrape,ok,chunk,2025-12-02T16:06:44,None,"{'2': {'processed': 1, 'failed': 0, 'discovered': 1, 'next_chunk_index': 1}, '4': {'processed': 25, 'failed': 0, 'discovered': 25, 'next_chunk_index': 2}, '6': {'processed': 660, 'failed': 0, 'discovered': 660, 'next_chunk_index': 33}, '8': {'processed': 0, 'failed': 0, 'discovered': 452, 'next_chunk_index': 0}}"
clean,pending,NaN,None,None,NaN
test_basic,pending,NaN,None,None,NaN
test_first_level,pending,NaN,None,None,NaN
test_duplicates,pending,NaN,None,None,NaN
fix,pending,NaN,None,None,NaN


In [11]:
df.df_peek()

,country,task,status,type,last_run,error,chunk_state
790,Afghanistan,test_duplicates,pending,NaN,None,None,NaN
788,Afghanistan,test_basic,pending,NaN,None,None,NaN
789,Afghanistan,test_first_level,pending,NaN,None,None,NaN
791,Afghanistan,fix,pending,NaN,None,None,NaN
787,Afghanistan,clean,pending,NaN,None,None,NaN
786,Afghanistan,scrape,ok,normal,2025-12-01T16:46:54,None,NaN
1402,AkrotiriAndDhekelia,test_duplicates,pending,NaN,None,None,NaN
1398,AkrotiriAndDhekelia,scrape,ok,normal,2025-12-01T16:46:54,None,NaN
1399,AkrotiriAndDhekelia,clean,pending,NaN,None,None,NaN
1400,AkrotiriAndDhekelia,test_basic,pending,NaN,None,None,NaN


## * check tasks state

In [203]:
scrape_ok = df.query("task == 'scrape' and status == 'ok'")["country"]
print(len(scrape_ok))

98


In [161]:
df.query("country in @ok_scrape").df_peek()

,country,task,status,type,last_run,error,chunk_state
787,Afghanistan,clean,pending,NaN,None,None,NaN
791,Afghanistan,fix,pending,NaN,None,None,NaN
786,Afghanistan,scrape,ok,normal,2025-12-02T16:06:44,None,NaN
788,Afghanistan,test_basic,pending,NaN,None,None,NaN
790,Afghanistan,test_duplicates,pending,NaN,None,None,NaN
789,Afghanistan,test_first_level,pending,NaN,None,None,NaN
1399,AkrotiriAndDhekelia,clean,pending,NaN,None,None,NaN
1403,AkrotiriAndDhekelia,fix,pending,NaN,None,None,NaN
1398,AkrotiriAndDhekelia,scrape,ok,normal,2025-12-02T16:06:44,None,NaN
1400,AkrotiriAndDhekelia,test_basic,pending,NaN,None,None,NaN


In [204]:
clean_pending = df.query("country in @ok_scrape").query("task == 'clean' and status == 'pending'")["country"]
print(len(clean_pending))

26


## * filter only sovereign countries

In [6]:
sovereign_countries = tgm.load(DATA_DIR / 'sovereign countries.json')
print(len(sovereign_countries))
process_state = tgm.load(DATA_DIR / "process_state.json")
print(len(process_state))

214
241


In [9]:
process_state_sovereign = {c:v for c,v in process_state.items() if c in sovereign_countries}
print(len(process_state_sovereign))

214


In [10]:
tgm.dump(DATA_DIR / "process_state.json", process_state_sovereign)